# SEC 10-K Sentence Labeling Pipeline - Google Colab

This notebook processes SEC 10-K Item extractions into sentence-level datasets ready for manual labeling.

**Workflow:**
1. Setup environment & mount Drive
2. Transform MDA format (if needed)
3. Build sentence table from Item extractions
4. Create balanced labeling sample
5. Download & label in Google Sheets
6. Process labeled data

**Data location:** `/content/drive/MyDrive/sec_10k_project/`

**Updated:** 2025-12-16 - Added automatic MDA format detection and transformation

## 📦 Step 1: Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set base path (CUSTOMIZE THIS)
BASE_PATH = '/content/drive/MyDrive/sec_10k_project'

print(f"✓ Drive mounted. Base path: {BASE_PATH}")

In [ ]:
# Create folder structure
import os

folders = [
    'extracted_items',
    'sentence_tables',
    'labeling_samples',
    'labeled_data'
]

for folder in folders:
    path = f'{BASE_PATH}/{folder}'
    os.makedirs(path, exist_ok=True)
    print(f"✓ {path}")

print("\n✓ Folder structure ready")

In [ ]:
# Install dependencies
!pip install -q pandas numpy tqdm pyyaml pyarrow
!pip install -q ftfy contractions chardet
!pip install -q --use-pep517 jieba
!pip install -q spacy
!python -m spacy download en_core_web_sm

print("✓ Dependencies installed")

In [ ]:
# Clone cntext repository
import os
if not os.path.exists('/content/cntext'):
    !git clone https://github.com/haowenluo/cntext.git /content/cntext
    print("✓ Cloned cntext repository")
else:
    print("✓ cntext repository already exists")

import sys
sys.path.insert(0, '/content/cntext')

# Install remaining cntext dependencies
!pip install -q networkx scipy scikit-learn gensim nltk opencc-python-reimplemented
!pip install -q distinctiveness aiolimiter instructor pydantic psutil

# Verify
import cntext as ct
print(f"✓ cntext version: {ct.__version__}")

In [ ]:
# Copy pipeline scripts from tech_adoption_project folder
!cp /content/cntext/tech_adoption_project/build_sentence_table.py /content/
!cp /content/cntext/tech_adoption_project/build_labeling_sample.py /content/
!cp /content/cntext/tech_adoption_project/tech_keywords.yaml /content/

print("✓ Pipeline scripts ready")
print("  - build_sentence_table.py")
print("  - build_labeling_sample.py")
print("  - tech_keywords.yaml")

## 🔄 Step 2: MDA Format Transformation (NEW!)

**This step automatically detects and transforms MDA JSON format to pipeline format.**

### Your MDA Format:
```json
{
  "cik": "1643988",
  "company": "Company_1643988",
  "filing_type": "10-K",
  "filing_date": "2020-01-01",
  "period_of_report": "2020-12-31",
  "filename": "1643988_10K_2020_0001387131-21-004517.htm",
  "item_7": "ITEM 7. MANAGEMENT'S DISCUSSION..."
}
```

### Pipeline Expected Format:
```json
{
  "cik": 1643988,
  "accession": "0001387131-21-004517",
  "fiscal_year": 2020,
  "filing_date": "2021-02-15",
  "item": "7",
  "item_text": "ITEM 7. MANAGEMENT'S DISCUSSION..."
}
```

In [ ]:
# MDA Format Transformation Functions
import json
import glob
from pathlib import Path

def detect_format(data):
    """
    Detect if data is in MDA format or pipeline format.
    
    Returns:
        'mda' or 'pipeline' or 'unknown'
    """
    if isinstance(data, dict):
        # Check for MDA format indicators
        if 'item_7' in data and 'period_of_report' in data:
            return 'mda'
        # Check for pipeline format indicators
        elif 'item_text' in data and 'accession' in data:
            return 'pipeline'
    return 'unknown'


def transform_mda_to_pipeline_format(mda_json_path):
    """
    Transform a single MDA JSON file to pipeline format.
    
    MDA format:
        cik, company, filing_type, filing_date, period_of_report, filename, item_7
    
    Pipeline format:
        cik, accession, fiscal_year, filing_date, item, item_text
    """
    with open(mda_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Extract accession from filename
    # e.g., "1643988_10K_2020_0001387131-21-004517.htm" -> "0001387131-21-004517"
    filename = data.get('filename', '')
    parts = filename.replace('.htm', '').replace('.json', '').split('_')
    accession = parts[-1] if len(parts) >= 4 else 'unknown'
    
    # Extract fiscal year from period_of_report
    # e.g., "2020-12-31" -> 2020
    period = data.get('period_of_report', '')
    fiscal_year = int(period[:4]) if period and len(period) >= 4 else 0
    
    # Get CIK as integer
    cik = data.get('cik', '0')
    cik = int(cik) if isinstance(cik, str) and cik.isdigit() else cik
    
    return {
        'cik': cik,
        'accession': accession,
        'fiscal_year': fiscal_year,
        'filing_date': data.get('filing_date', ''),
        'item': '7',  # MD&A is always Item 7
        'item_text': data.get('item_7', '')
    }


def process_mda_directory(input_dir, output_file):
    """
    Process all MDA JSON files in a directory and its subdirectories.
    Transforms them to pipeline format and saves as a single combined JSON.
    
    Args:
        input_dir: Directory containing MDA JSON files
        output_file: Output path for combined JSON file
    
    Returns:
        List of transformed records
    """
    all_records = []
    json_files = list(Path(input_dir).rglob("*.json"))
    
    # Filter out non-MDA files
    json_files = [f for f in json_files if f.name not in ['sampling_log.csv', 'sampling_summary.txt']]
    
    print(f"Found {len(json_files)} JSON files")
    
    transformed_count = 0
    skipped_count = 0
    
    for filepath in json_files:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Detect format
            fmt = detect_format(data)
            
            if fmt == 'mda':
                # Transform MDA format
                record = transform_mda_to_pipeline_format(filepath)
                if record['item_text']:  # Skip empty MD&A
                    all_records.append(record)
                    transformed_count += 1
                    if transformed_count <= 5:  # Show first 5
                        print(f"  ✓ {filepath.name} (CIK: {record['cik']}, Year: {record['fiscal_year']})")
                else:
                    print(f"  ⚠ {filepath.name} - Empty item_7, skipped")
                    skipped_count += 1
            elif fmt == 'pipeline':
                # Already in correct format
                if data.get('item_text'):
                    all_records.append(data)
                    transformed_count += 1
                else:
                    skipped_count += 1
            else:
                print(f"  ⚠ {filepath.name} - Unknown format, skipped")
                skipped_count += 1
                
        except Exception as e:
            print(f"  ✗ Error processing {filepath.name}: {e}")
            skipped_count += 1
    
    if transformed_count > 5:
        print(f"  ... ({transformed_count - 5} more files processed)")
    
    # Save combined file
    if all_records:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_records, f, indent=2, ensure_ascii=False)
        print(f"\n✓ Saved combined file: {output_file}")
        print(f"  Total records: {len(all_records)}")
        print(f"  Transformed: {transformed_count}")
        print(f"  Skipped: {skipped_count}")
    else:
        print(f"\n✗ No valid records found!")
    
    return all_records

print("✓ MDA transformation functions loaded")

In [ ]:
# List available input files
import os

input_dir = f'{BASE_PATH}/extracted_items'

# Check for JSON files
json_files = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith('.json'):
            json_files.append(os.path.join(root, f))

print(f"Available JSON files in {input_dir}:")
print(f"  Found {len(json_files)} JSON files")

if json_files:
    # Show first few
    for i, f in enumerate(json_files[:5]):
        rel_path = os.path.relpath(f, input_dir)
        print(f"  [{i}] {rel_path}")
    if len(json_files) > 5:
        print(f"  ... ({len(json_files) - 5} more files)")
    
    # Detect format of first file
    print(f"\nDetecting format from first file...")
    with open(json_files[0], 'r') as f:
        sample_data = json.load(f)
    fmt = detect_format(sample_data)
    print(f"  Format detected: {fmt}")
    
    if fmt == 'mda':
        print(f"  ✓ MDA format detected - transformation will be applied")
        print(f"  Fields found: {list(sample_data.keys())}")
    elif fmt == 'pipeline':
        print(f"  ✓ Pipeline format detected - no transformation needed")
    else:
        print(f"  ⚠ Unknown format - please check your data structure")
else:
    print("\n⚠️  No JSON files found! Please upload your Item extractions first.")

In [ ]:
# Transform MDA files to pipeline format
# Run this cell if you have MDA format files

print("="*80)
print("TRANSFORMING MDA FILES TO PIPELINE FORMAT")
print("="*80)

input_directory = f'{BASE_PATH}/extracted_items'
output_combined = f'{BASE_PATH}/extracted_items/items_combined.json'

records = process_mda_directory(input_directory, output_combined)

if records:
    print(f"\n✓ Transformation complete!")
    print(f"  Use this file for next step: items_combined.json")
    
    # Set INPUT_FILENAME for next cells
    INPUT_FILENAME = 'items_combined.json'
else:
    print(f"\n⚠ No records transformed. Check your input files.")

## 📝 Step 3: Build Sentence Table

**Before running:** If you haven't run the transformation step above, make sure your files are in pipeline format:

```json
{
  "cik": 1234567,
  "accession": "0000000000-20-000001",
  "fiscal_year": 2020,
  "filing_date": "2021-02-15",
  "item": "1",
  "item_text": "Your extracted text..."
}
```

In [ ]:
# CONFIGURE: Set your input file name (if not using transformed file)
# INPUT_FILENAME = 'items_2020.json'  # ← Uncomment and change if needed

# INPUT_FILENAME should be set from transformation step above
print(f"Input filename: {INPUT_FILENAME}")

# Run sentence table builder
from build_sentence_table import build_sentence_table

input_file = f'{BASE_PATH}/extracted_items/{INPUT_FILENAME}'
output_file = f'{BASE_PATH}/sentence_tables/sentence_table_{INPUT_FILENAME.split(".")[0]}.csv'

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print("\nProcessing...\n")

sentence_df = build_sentence_table(
    input_path=input_file,
    output_path=output_file,
    output_format='both'
)

print(f"\n✓ Sentence table created: {len(sentence_df):,} sentences")

In [ ]:
# Verify output
import pandas as pd

df = pd.read_csv(output_file)

print(f"Total sentences: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSentences by Item:")
print(df['item'].value_counts().sort_index())
print(f"\nSentences by Year:")
print(df['fiscal_year'].value_counts().sort_index())

print(f"\nSample sentences:")
print(df[['cik', 'fiscal_year', 'item', 'sentence_text']].head(5))

## 🎯 Step 4: Build Labeling Sample

Creates a balanced sample of ~2500 sentences:
- 50% from tech keyword hits
- 50% from random background

In [ ]:
# (Optional) Customize tech keywords
import yaml

# View current keywords
with open('/content/tech_keywords.yaml', 'r') as f:
    keywords = yaml.safe_load(f)

print("Current keyword categories:")
for category in keywords['Dictionary'].keys():
    count = len(keywords['Dictionary'][category])
    print(f"  {category}: {count} terms")

# Add custom keywords if needed
# keywords['Dictionary']['custom_category'] = ['term1', 'term2', ...]
# with open('/content/tech_keywords.yaml', 'w') as f:
#     yaml.dump(keywords, f)

In [ ]:
# Run labeling sample builder
from build_labeling_sample import build_labeling_sample
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
tqdm.pandas()

# Configure sample size (adjust based on your data size)
# For small datasets (<100 files), use 500-1000
# For large datasets (>1000 files), use 2500-5000
SAMPLE_SIZE = min(2500, len(sentence_df))  # ← Auto-adjust or manually set

input_file = output_file  # Use sentence table from Step 3
label_output = f'{BASE_PATH}/labeling_samples/label_set_{INPUT_FILENAME.split(".")[0]}.csv'

print(f"Input:  {input_file}")
print(f"Output: {label_output}")
print(f"Sample size: {SAMPLE_SIZE}")
print("\nProcessing...\n")

label_df = build_labeling_sample(
    input_path=input_file,
    output_path=label_output,
    keywords_file='/content/tech_keywords.yaml',
    sample_size=SAMPLE_SIZE
)

print(f"\n✓ Labeling sample created: {len(label_df):,} sentences")

In [ ]:
# Review sample quality
print("Source pool distribution:")
print(label_df['source_pool'].value_counts())

print("\nTech hit distribution:")
print(label_df['tech_hit'].value_counts())

print("\n" + "="*80)
print("Sample TECH HITS:")
print("="*80)
tech_samples = label_df[label_df['tech_hit'] == True].head(10)
for idx, row in tech_samples.iterrows():
    print(f"\n[{idx}] {row['sentence_text'][:200]}...")

print("\n" + "="*80)
print("Sample RANDOM:")
print("="*80)
random_samples = label_df[label_df['source_pool'] == 'random'].head(10)
for idx, row in random_samples.iterrows():
    print(f"\n[{idx}] {row['sentence_text'][:200]}...")

## 📥 Step 5: Download for Labeling

**Option A: Open in Google Sheets (Recommended)**
1. Navigate to folder in Drive: `sec_10k_project/labeling_samples/`
2. Right-click CSV → "Open with" → "Google Sheets"
3. Label directly in Sheets (auto-saves!)

**Option B: Download to local**

In [ ]:
# Download to local machine
from google.colab import files

print(f"Downloading: {label_output}")
files.download(label_output)

## ✏️ Labeling Instructions

For each sentence, mark **EXACTLY ONE** column with `1` (leave others blank or `0`):

| Column | Description | Examples |
|--------|-------------|----------|
| `TECH_IMPL` | Technology implementation/usage | "We deployed AI algorithms", "Our cloud infrastructure processes..." |
| `TECH_ADOPT` | Technology adoption/investment | "We invested $50M in R&D", "We acquired a ML company" |
| `TECH_PRODUCT` | Technology product/offering | "Our SaaS platform offers...", "We sell cybersecurity software" |
| `NON_TECH` | Not technology-related | "We operate retail stores", "Revenue increased 10%" |

**Constraint:** If `NON_TECH=1`, all other columns must be `0`

**Tips:**
- `tech_hit` column is just a hint (not always accurate)
- Focus on sentence's **main topic**
- Be consistent across similar sentences

## ✅ Step 6: Validate Labeled Data

**After labeling:** Save as CSV and upload to `sec_10k_project/labeled_data/`

In [ ]:
# Load labeled data
LABELED_FILENAME = 'labeled_set_items_combined.csv'  # ← CHANGE THIS

labeled_file = f'{BASE_PATH}/labeled_data/{LABELED_FILENAME}'
labeled_df = pd.read_csv(labeled_file)

print(f"Loaded {len(labeled_df):,} labeled sentences")

In [ ]:
# Validate labels
label_cols = ['TECH_IMPL', 'TECH_ADOPT', 'TECH_PRODUCT', 'NON_TECH']

print("Label distribution:")
for col in label_cols:
    count = (labeled_df[col] == 1).sum()
    pct = count / len(labeled_df) * 100
    print(f"  {col}: {count:,} ({pct:.1f}%)")

# Quality checks
label_sum = labeled_df[label_cols].sum(axis=1)

unlabeled = (label_sum == 0).sum()
multi_label = (label_sum > 1).sum()
valid = (label_sum == 1).sum()

print(f"\nQuality checks:")
print(f"  ✓ Valid (exactly 1 label): {valid:,} ({valid/len(labeled_df)*100:.1f}%)")
print(f"  ⚠ Unlabeled (0 labels): {unlabeled:,}")
print(f"  ❌ Multi-label (>1 labels): {multi_label:,}")

if multi_label > 0:
    print("\n⚠️  Found multi-label rows (should fix):")
    bad_rows = labeled_df[label_sum > 1][['cik', 'sentence_text'] + label_cols].head(5)
    print(bad_rows)

In [ ]:
# Export clean labeled data
clean_df = labeled_df[label_sum == 1].copy()
clean_output = f'{BASE_PATH}/labeled_data/clean_{LABELED_FILENAME}'

clean_df.to_csv(clean_output, index=False)
print(f"✓ Saved clean labeled data: {len(clean_df):,} sentences")
print(f"  {clean_output}")

## 📊 Next Steps: Train Classifier

Now you have clean labeled data ready for ML training!

**Suggested approaches:**
1. **Simple baseline**: Logistic Regression with TF-IDF
2. **Better performance**: Fine-tune BERT/RoBERTa
3. **Zero-shot**: Use LLM (GPT-4, Claude) via cntext's LLM module

The labeled data is in your Drive at:
`/content/drive/MyDrive/sec_10k_project/labeled_data/`